# CatBoost Road-Only 4.0 Positive vs Negative Feature Analysis

Notebook сравнивает строки `target=1` и `target=0` для признаков модели `road_only_4_0`.

Здесь считаются:

- `mean_pos / mean_neg`;
- `median_pos / median_neg`;
- `p95_pos / p95_neg`;
- `zero_share_pos / zero_share_neg`;
- KS statistic;
- Mann-Whitney p-value;
- single-feature ROC-AUC;
- standardized mean difference.

Анализ считается на train-части road-only строк:

```text
month < 2023-01-01
road_length_total_m > 0
```

Для 4.0 применяется `p99 clipping`. Для 4.1 применяется `p99 clipping + log1p`
ровно к тем признакам, которые использовались в модели.

## 1. Запуск анализа

Вся логика вынесена в `scripts/positive_negative_feature_analysis.py`,
чтобы notebook и batch-отчет не расходились.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT.parent / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from positive_negative_feature_analysis import run_analysis

result = run_analysis("road_only_4_0", show_plots=True)
comparison = result["comparison"]
train = result["train"]

## 2. Главная таблица

Сортировка по `single_feature_auc_distance`: насколько один признак сам по себе
отделяет `target=1` от `target=0`.

In [ ]:
display(
    comparison[
        [
            "feature",
            "importance",
            "transformed_log1p",
            "single_feature_roc_auc",
            "single_feature_oriented_auc",
            "auc_direction",
            "ks_statistic",
            "mannwhitney_pvalue",
            "standardized_mean_difference",
            "mean_pos",
            "mean_neg",
            "mean_ratio_pos_to_neg",
            "median_pos",
            "median_neg",
            "p95_pos",
            "p95_neg",
            "zero_share_pos",
            "zero_share_neg",
        ]
    ].head(40)
)

## 3. Неочевидные случаи

Эти признаки важны для CatBoost, но слабее выглядят как одиночные разделители.
Значит, они могут работать через взаимодействия с другими признаками.

In [ ]:
interaction_like = comparison.dropna(subset=["importance", "single_feature_auc_distance"]).copy()
interaction_like = interaction_like[
    (interaction_like["importance"] >= interaction_like["importance"].quantile(0.70))
    & (interaction_like["single_feature_auc_distance"] <= interaction_like["single_feature_auc_distance"].quantile(0.50))
].sort_values("importance", ascending=False)

display(
    interaction_like[
        [
            "feature",
            "importance",
            "single_feature_roc_auc",
            "ks_statistic",
            "standardized_mean_difference",
            "zero_share_pos",
            "zero_share_neg",
        ]
    ].head(30)
)

## 4. Где лежат сохраненные артефакты

In [ ]:
result["summary"]